# Phase 3 Policy Behavior Review

Read-only visual review of the selected turnover-v2 policy artifacts. The notebook displays observed target allocations, cross-seed dispersion, equity-like and SHY exposure, turnover, high-volatility periods, and counterfactual sensitivity. It does not retrain models or access the test split.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "artifacts").exists() else cwd.parent
DIAGNOSTICS = ROOT / "artifacts/diagnostics/ppo_phase3_seed_sweep_turnover_v2"
SENSITIVITY = ROOT / "artifacts/sensitivity/ppo_phase3_seed_sweep_turnover_v2"
STATISTICAL_VALIDATION = ROOT / "artifacts/statistical_validation/ppo_phase3_seed_sweep_turnover_v2"
OUTPUT_DIR = ROOT / "artifacts/visualizations/ppo_phase3_seed_sweep_turnover_v2"

paths = {
    "allocations": DIAGNOSTICS / "allocation_by_regime.parquet",
    "nav": DIAGNOSTICS / "nav_by_regime.parquet",
    "turnover": DIAGNOSTICS / "turnover_distribution.parquet",
    "sensitivity": SENSITIVITY / "sensitivity_summary.csv",
    "bootstrap_summary": STATISTICAL_VALIDATION / "bootstrap_summary.json",
    "bootstrap_samples": STATISTICAL_VALIDATION / "bootstrap_samples.parquet",
    "equal_weight": ROOT / "artifacts/backtests/baselines_validation_turnover_v2/equal_weight_weekly/nav.parquet",
}
missing = [name for name, path in paths.items() if not path.exists()]
assert not missing, f"Missing Phase 3 artifacts: {missing}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titleweight": "bold"})
ROOT

In [ ]:
allocations = pd.read_parquet(paths["allocations"])
nav = pd.read_parquet(paths["nav"])
turnover = pd.read_parquet(paths["turnover"])
sensitivity = pd.read_csv(paths["sensitivity"])
bootstrap_summary = json.loads(paths["bootstrap_summary"].read_text())
bootstrap_samples = pd.read_parquet(paths["bootstrap_samples"])
equal_weight = pd.read_parquet(paths["equal_weight"])

allocations["date"] = pd.to_datetime(allocations["date"])
nav["date"] = pd.to_datetime(nav["date"])
turnover["date"] = pd.to_datetime(turnover["date"])
equal_weight["date"] = pd.to_datetime(equal_weight["date"])

assert not (allocations["split"] == "test").any()
assert not (nav["split"] == "test").any()
assert not (turnover["split"] == "test").any()
assert bootstrap_summary["test_split_used"] is False
print(f"Allocation rows: {len(allocations):,}")
print(f"Seeds: {sorted(allocations.seed.unique())}")
print(f"Regimes: {sorted(allocations.regime_name.unique())}")

## Review Controls

Change `SEED` or `REGIME`, then rerun the cells below. The defaults show seed 42 during the out-of-sample 2024 validation window.

In [ ]:
SEED = 42
REGIME = "validation_2024"

assert SEED in set(allocations.seed)
assert REGIME in set(allocations.regime_name)
SEED, REGIME

In [ ]:
ASSET_ORDER = [
    "SPY",
    "QQQ",
    "IWM",
    "EFA",
    "EEM",
    "TLT",
    "IEF",
    "SHY",
    "LQD",
    "HYG",
    "GLD",
    "DBC",
    "VNQ",
    "XLU",
]
EQUITY_LIKE = {"SPY", "QQQ", "IWM", "EFA", "EEM", "VNQ", "XLU"}
palette = plt.get_cmap("tab20")
COLORS = {ticker: palette(index) for index, ticker in enumerate(ASSET_ORDER)}


def save_and_show(fig, filename):
    path = OUTPUT_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.show()
    print(path.relative_to(ROOT))


def shade_high_volatility(ax, regime):
    flags = (
        allocations.loc[allocations.regime_name == regime, ["date", "high_volatility"]]
        .drop_duplicates()
        .sort_values("date")
    )
    for date in flags.loc[flags.high_volatility, "date"]:
        ax.axvspan(
            date, date + pd.Timedelta(days=7), color="crimson", alpha=0.06, linewidth=0
        )

## Selected Seed: Target Allocation Through Time

Each vertical slice sums to 100%. Red shading marks the top quartile of normalized SPY 21-day volatility within the selected regime.

In [ ]:
selected = allocations[(allocations.seed == SEED) & (allocations.regime_name == REGIME)]
weights = selected.pivot(
    index="date", columns="ticker", values="target_weight"
).reindex(columns=ASSET_ORDER)

fig, ax = plt.subplots(figsize=(16, 7))
ax.stackplot(
    weights.index,
    weights.T,
    labels=weights.columns,
    colors=[COLORS[t] for t in weights.columns],
    alpha=0.9,
)
shade_high_volatility(ax, REGIME)
ax.set(
    title=f"Seed {SEED}: target allocation — {REGIME}",
    ylabel="Portfolio weight",
    ylim=(0, 1),
)
ax.legend(ncol=7, loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False)
save_and_show(fig, f"seed_{SEED}_{REGIME}_allocation.png")

## Validation Performance vs. Equal Weight

All strategies start from the same equal-weight portfolio and initial NAV of 1.0. The benchmark rebalances weekly and uses the same corrected one-way-turnover definition and 10 bps transaction-cost convention as the selected PPO runs. The shaded cross-seed range is descriptive, not a confidence interval.

In [ ]:
VALIDATION_REGIME = "validation_2024"
validation_nav = nav[nav.regime_name == VALIDATION_REGIME].copy()
assert set(validation_nav.selection_checkpoint) == {"best_checkpoint"}
assert set(validation_nav.split) == {"validation"}

agent_wealth = validation_nav.pivot(index="date", columns="seed", values="nav").sort_index()
benchmark = equal_weight.set_index("date").sort_index()
assert agent_wealth.index.equals(benchmark.index)
assert not agent_wealth.isna().any().any()
assert np.isfinite(agent_wealth.to_numpy()).all()


def performance_row(label, daily_returns, benchmark_returns):
    daily_returns = daily_returns.astype(float)
    benchmark_returns = benchmark_returns.astype(float)
    wealth = (1.0 + daily_returns).cumprod()
    drawdown = wealth / wealth.cummax().clip(lower=1.0) - 1.0
    volatility = daily_returns.std(ddof=1) * np.sqrt(252)
    sharpe = daily_returns.mean() / daily_returns.std(ddof=1) * np.sqrt(252)
    active_returns = daily_returns - benchmark_returns
    tracking_error = active_returns.std(ddof=1) * np.sqrt(252)
    information_ratio = (
        active_returns.mean() / active_returns.std(ddof=1) * np.sqrt(252)
        if tracking_error > 0
        else np.nan
    )
    monthly = (1.0 + daily_returns).groupby(daily_returns.index.to_period("M")).prod() - 1.0
    benchmark_monthly = (
        (1.0 + benchmark_returns)
        .groupby(benchmark_returns.index.to_period("M"))
        .prod()
        - 1.0
    )
    return {
        "strategy": label,
        "total_return": wealth.iloc[-1] - 1.0,
        "annualized_volatility": volatility,
        "sharpe_ratio": sharpe,
        "max_drawdown": drawdown.min(),
        "tracking_error": tracking_error,
        "information_ratio": information_ratio,
        "monthly_outperformance_rate": (
            (monthly > benchmark_monthly).mean() if tracking_error > 0 else np.nan
        ),
    }


benchmark_returns = benchmark["daily_return"]
metric_rows = [
    performance_row(
        f"PPO seed {seed}",
        validation_nav.loc[validation_nav.seed == seed].set_index("date")["daily_return"],
        benchmark_returns,
    )
    for seed in agent_wealth.columns
]
seed_metrics = pd.DataFrame(metric_rows).set_index("strategy")
median_metrics = seed_metrics.median().rename("PPO seed median")
benchmark_metrics = pd.Series(
    performance_row("Equal weight weekly", benchmark_returns, benchmark_returns)
).drop("strategy")
comparison_metrics = pd.concat(
    [seed_metrics, median_metrics.to_frame().T, benchmark_metrics.to_frame().T]
)
comparison_metrics.style.format(
    {
        "total_return": "{:.2%}",
        "annualized_volatility": "{:.2%}",
        "sharpe_ratio": "{:.2f}",
        "max_drawdown": "{:.2%}",
        "tracking_error": "{:.2%}",
        "information_ratio": "{:.2f}",
        "monthly_outperformance_rate": "{:.2%}",
    },
    na_rep="—",
)

In [ ]:
fig, axes = plt.subplots(
    2,
    1,
    figsize=(16, 9),
    sharex=True,
    gridspec_kw={"height_ratios": [2, 1]},
)
seed_min = agent_wealth.min(axis=1)
seed_median = agent_wealth.median(axis=1)
seed_max = agent_wealth.max(axis=1)
axes[0].fill_between(
    agent_wealth.index,
    seed_min,
    seed_max,
    color="#4c78a8",
    alpha=0.12,
    label="PPO seed range (not CI)",
)
for seed in agent_wealth.columns:
    axes[0].plot(
        agent_wealth.index,
        agent_wealth[seed],
        color="#4c78a8",
        alpha=0.32,
        linewidth=1,
    )
axes[0].plot(
    seed_median.index,
    seed_median,
    color="#1f4e79",
    linewidth=2.5,
    label="PPO seed median",
)
axes[0].plot(
    benchmark.index,
    benchmark["nav"],
    color="black",
    linestyle="--",
    linewidth=2,
    label="Equal weight weekly",
)
axes[0].set(title="Selected PPO checkpoints vs. equal weight — validation 2024", ylabel="NAV")
axes[0].legend(frameon=False, ncol=3)

active_wealth = agent_wealth.div(benchmark["nav"], axis=0) - 1.0
for seed in active_wealth.columns:
    axes[1].plot(
        active_wealth.index,
        active_wealth[seed],
        color="#4c78a8",
        alpha=0.32,
        linewidth=1,
    )
axes[1].plot(
    active_wealth.index,
    active_wealth.median(axis=1),
    color="#1f4e79",
    linewidth=2.5,
    label="Median active wealth",
)
axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].set(xlabel="Date", ylabel="PPO / equal weight − 1")
axes[1].legend(frameon=False)
fig.tight_layout()
save_and_show(fig, "validation_2024_ppo_vs_equal_weight.png")

In [ ]:
selected_validation = allocations[
    (allocations.seed == SEED) & (allocations.regime_name == VALIDATION_REGIME)
]
selected_weights = selected_validation.pivot(
    index="date", columns="ticker", values="target_weight"
).reindex(columns=ASSET_ORDER)
selected_wealth = agent_wealth[SEED]

fig, axes = plt.subplots(
    2,
    1,
    figsize=(16, 9),
    sharex=True,
    gridspec_kw={"height_ratios": [2, 1]},
)
axes[0].stackplot(
    selected_weights.index,
    selected_weights.T,
    labels=selected_weights.columns,
    colors=[COLORS[ticker] for ticker in selected_weights.columns],
    alpha=0.9,
)
shade_high_volatility(axes[0], VALIDATION_REGIME)
axes[0].set(title=f"Seed {SEED}: allocation decisions and realized wealth — validation 2024", ylabel="Target weight", ylim=(0, 1))
axes[0].legend(ncol=7, loc="upper center", bbox_to_anchor=(0.5, -0.10), frameon=False)
axes[1].plot(selected_wealth.index, selected_wealth, linewidth=2.2, label=f"PPO seed {SEED}")
axes[1].plot(
    benchmark.index,
    benchmark["nav"],
    color="black",
    linestyle="--",
    linewidth=1.8,
    label="Equal weight weekly",
)
shade_high_volatility(axes[1], VALIDATION_REGIME)
axes[1].set(xlabel="Date", ylabel="NAV")
axes[1].legend(frameon=False)
fig.tight_layout()
save_and_show(fig, f"seed_{SEED}_validation_2024_allocation_and_wealth.png")

## Paired Bootstrap Evidence

The forest plot shows observed active return and paired 95% moving-block-bootstrap intervals. The campaign result is the median across the five policy seeds within each bootstrap replication. Intervals describe validation stability and do not remove model-selection bias.

In [ ]:
bootstrap_rows = []
for group in bootstrap_summary["groups"]:
    result = group["metrics"]["active_total_return"]
    bootstrap_rows.append(
        {
            "label": (
                "Campaign median"
                if group["aggregation"] == "campaign_median"
                else f"Seed {group['seed']}"
            ),
            "observed": result["observed"],
            "lower": result["confidence_interval_lower"],
            "upper": result["confidence_interval_upper"],
            "probability_positive": result["probability_positive"],
        }
    )
bootstrap_evidence = pd.DataFrame(bootstrap_rows)
campaign_samples = bootstrap_samples[
    bootstrap_samples.aggregation == "campaign_median"
]["active_total_return"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
y = np.arange(len(bootstrap_evidence))
observed_pct = 100 * bootstrap_evidence.observed
lower_error = 100 * (bootstrap_evidence.observed - bootstrap_evidence.lower)
upper_error = 100 * (bootstrap_evidence.upper - bootstrap_evidence.observed)
axes[0].errorbar(
    observed_pct,
    y,
    xerr=np.vstack([lower_error, upper_error]),
    fmt="o",
    color="#1f4e79",
    ecolor="#4c78a8",
    capsize=4,
)
axes[0].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[0].set_yticks(y, bootstrap_evidence.label)
axes[0].invert_yaxis()
axes[0].set(
    title="Observed active return with paired 95% intervals",
    xlabel="PPO minus equal-weight total return (percentage points)",
)

axes[1].hist(
    100 * campaign_samples,
    bins=50,
    color="#4c78a8",
    alpha=0.75,
)
campaign_observed = bootstrap_evidence.loc[
    bootstrap_evidence.label == "Campaign median", "observed"
].iloc[0]
axes[1].axvline(0.0, color="black", linestyle="--", linewidth=1, label="No active return")
axes[1].axvline(
    100 * campaign_observed,
    color="#1f4e79",
    linewidth=2,
    label="Observed campaign median",
)
campaign_probability = bootstrap_evidence.loc[
    bootstrap_evidence.label == "Campaign median", "probability_positive"
].iloc[0]
axes[1].set(
    title=f"Campaign median bootstrap distribution — P(positive)={campaign_probability:.1%}",
    xlabel="PPO minus equal-weight total return (percentage points)",
    ylabel="Bootstrap replications",
)
axes[1].legend(frameon=False)
fig.tight_layout()
save_and_show(fig, "validation_2024_bootstrap_evidence.png")

bootstrap_evidence.style.format(
    {
        "observed": "{:.2%}",
        "lower": "{:.2%}",
        "upper": "{:.2%}",
        "probability_positive": "{:.1%}",
    }
)

## Cross-Seed Allocation Stability

Each panel shows the median asset weight across seeds with a 10th–90th percentile band.

In [ ]:
regime_allocations = allocations[allocations.regime_name == REGIME]
quantiles = (
    regime_allocations.groupby(["date", "ticker"])["target_weight"]
    .quantile([0.1, 0.5, 0.9])
    .unstack()
    .rename(columns={0.1: "q10", 0.5: "median", 0.9: "q90"})
    .reset_index()
)

fig, axes = plt.subplots(4, 4, figsize=(16, 12), sharex=True, sharey=True)
for ax, ticker in zip(axes.flat, ASSET_ORDER, strict=False):
    asset = quantiles[quantiles.ticker == ticker]
    ax.fill_between(asset.date, asset.q10, asset.q90, color=COLORS[ticker], alpha=0.22)
    ax.plot(asset.date, asset["median"], color=COLORS[ticker], linewidth=1.8)
    ax.set_title(ticker)
    ax.set_ylim(0, max(0.16, quantiles.q90.max() * 1.08))
for ax in axes.flat[len(ASSET_ORDER) :]:
    ax.axis("off")
fig.suptitle(f"Cross-seed target weights — {REGIME}", fontsize=16, fontweight="bold")
fig.supxlabel("Decision date")
fig.supylabel("Portfolio weight")
fig.tight_layout()
save_and_show(fig, f"cross_seed_{REGIME}_asset_bands.png")

## Equity-Like and SHY Exposure

Solid lines are cross-seed medians; bands show the 10th–90th percentile range.

In [ ]:
exposure = (
    regime_allocations.assign(
        equity_component=lambda frame: (
            frame.target_weight * frame.ticker.isin(EQUITY_LIKE)
        ),
        shy_component=lambda frame: frame.target_weight * (frame.ticker == "SHY"),
    )
    .groupby(["seed", "date"], as_index=False)
    .agg(
        equity_like=("equity_component", "sum"),
        shy=("shy_component", "sum"),
    )
)

fig, ax = plt.subplots(figsize=(16, 6))
for column, color, label in [
    ("equity_like", "#1f77b4", "Equity-like"),
    ("shy", "#ff7f0e", "SHY"),
]:
    band = exposure.groupby("date")[column].quantile([0.1, 0.5, 0.9]).unstack()
    ax.fill_between(band.index, band[0.1], band[0.9], color=color, alpha=0.18)
    ax.plot(band.index, band[0.5], color=color, linewidth=2, label=label)
shade_high_volatility(ax, REGIME)
ax.set(
    title=f"Cross-seed exposure — {REGIME}", ylabel="Portfolio weight", ylim=(0, 0.75)
)
ax.legend(frameon=False)
save_and_show(fig, f"cross_seed_{REGIME}_equity_shy_exposure.png")

## One-Way Turnover Through Time

Turnover uses the corrected definition: `0.5 × sum(abs(target − pre-trade weights))`.

In [ ]:
regime_turnover = turnover[turnover.regime_name == REGIME]
turnover_band = (
    regime_turnover.groupby("date")["turnover"].quantile([0.1, 0.5, 0.9]).unstack()
)

fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(
    turnover_band.index,
    turnover_band[0.1],
    turnover_band[0.9],
    color="#9467bd",
    alpha=0.2,
)
ax.plot(
    turnover_band.index,
    turnover_band[0.5],
    color="#6f42a5",
    linewidth=2,
    label="Median turnover",
)
ax.axhline(
    0.50, color="black", linestyle="--", linewidth=1, label="Selection limit (50%)"
)
shade_high_volatility(ax, REGIME)
ax.set(
    title=f"Cross-seed one-way turnover — {REGIME}",
    ylabel="Weekly turnover",
    ylim=(0, max(0.55, turnover_band[0.9].max() * 1.1)),
)
ax.legend(frameon=False)
save_and_show(fig, f"cross_seed_{REGIME}_turnover.png")

## Counterfactual Sensitivity

High-minus-low risk exposure changes by seed for the selected regime.

In [ ]:
scenario = sensitivity[sensitivity.regime_name == REGIME].copy()
x = np.arange(len(scenario.seed.unique()))
width = 0.18
fig, ax = plt.subplots(figsize=(13, 5))
for offset, (probe, metric, color, label) in enumerate(
    [
        (
            "spy_volatility",
            "median_equity_like_weight_delta",
            "#1f77b4",
            "SPY vol: equity Δ",
        ),
        ("spy_volatility", "median_shy_weight_delta", "#9ecae1", "SPY vol: SHY Δ"),
        (
            "global_risk",
            "median_equity_like_weight_delta",
            "#d62728",
            "Global risk: equity Δ",
        ),
        ("global_risk", "median_shy_weight_delta", "#ff9896", "Global risk: SHY Δ"),
    ]
):
    values = scenario[scenario.probe == probe].sort_values("seed")
    ax.bar(x + (offset - 1.5) * width, values[metric], width, color=color, label=label)
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(0.01, color="gray", linestyle="--", linewidth=0.8)
ax.axhline(-0.01, color="gray", linestyle="--", linewidth=0.8)
ax.set_xticks(x, sorted(scenario.seed.unique()))
ax.set(
    title=f"Counterfactual high-minus-low risk response — {REGIME}",
    xlabel="Seed",
    ylabel="Weight change",
)
ax.legend(ncol=2, frameon=False)
save_and_show(fig, f"cross_seed_{REGIME}_sensitivity.png")

## Pre-Rendered Default Review

These images show the default seed 42 / validation 2024 review even before the notebook is executed.

![Paired bootstrap evidence](../artifacts/visualizations/ppo_phase3_seed_sweep_turnover_v2/validation_2024_bootstrap_evidence.png)

![PPO versus equal weight](../artifacts/visualizations/ppo_phase3_seed_sweep_turnover_v2/validation_2024_ppo_vs_equal_weight.png)

![Seed 42 allocation and wealth](../artifacts/visualizations/ppo_phase3_seed_sweep_turnover_v2/seed_42_validation_2024_allocation_and_wealth.png)

![Seed 42 allocation](../artifacts/visualizations/ppo_phase3_seed_sweep_turnover_v2/seed_42_validation_2024_allocation.png)

![Cross-seed asset bands](../artifacts/visualizations/ppo_phase3_seed_sweep_turnover_v2/cross_seed_validation_2024_asset_bands.png)

![Equity and SHY exposure](../artifacts/visualizations/ppo_phase3_seed_sweep_turnover_v2/cross_seed_validation_2024_equity_shy_exposure.png)

![Turnover](../artifacts/visualizations/ppo_phase3_seed_sweep_turnover_v2/cross_seed_validation_2024_turnover.png)

![Sensitivity](../artifacts/visualizations/ppo_phase3_seed_sweep_turnover_v2/cross_seed_validation_2024_sensitivity.png)

## Reading Guide

- The bootstrap forest plot distinguishes a positive point estimate from statistical uncertainty; intervals crossing zero are not conclusive evidence of outperformance.
- The validation wealth chart compares every selected best checkpoint with a cost-matched weekly equal-weight portfolio; its lower panel shows cumulative active wealth.
- The combined seed view aligns target allocations with realized PPO and equal-weight NAV paths.
- The stacked chart shows what one selected checkpoint actually held through time.
- The small multiples show whether asset allocations are consistent across seeds.
- The exposure and turnover charts highlight behavior during high-volatility weeks.
- The sensitivity chart separates historical association from direct counterfactual response.
- The 2020 and 2022 windows are in-sample diagnostics; `validation_2024` is the out-of-sample validation window. The test split is not used.